## Further Preprocessing for BiLSTM model

In [58]:
import numpy as np
import pandas as pd
import torch
from collections import Counter
from itertools import chain
import torch.nn as nn

In [39]:
# Load preprocessed input (Final_Cleaned_Tweet only)
X_train = pd.read_csv('X_train.csv', usecols=['Final_Cleaned_Tweet'])
X_val   = pd.read_csv('X_val.csv', usecols=['Final_Cleaned_Tweet'])
X_test  = pd.read_csv('X_test.csv', usecols=['Final_Cleaned_Tweet'])

# Load labels
y_train = pd.read_csv('y_train.csv').squeeze()
y_val   = pd.read_csv('y_val.csv').squeeze()
y_test  = pd.read_csv('y_test.csv').squeeze()

In [40]:
print(X_train.shape, y_train.shape)
print(X_train['Final_Cleaned_Tweet'].iloc[0])

(183861, 1) (183861,)
ottawa police chief swears repeat last winter freedom convoy pkbnews


### Build the Vocabulary from Training Data
***text is already cleaned and lemmatized, Now split train data on spaces***

### Tokenization and Vocabulary

In [ ]:
# Tokenize training tweets
token_lists = X_train['Final_Cleaned_Tweet'].astype(str).apply(lambda x: x.split()).tolist()
all_tokens = list(chain.from_iterable(token_lists))
token_freq = Counter(all_tokens)

# Vocab setup
MAX_VOCAB = 20000
specials = ['<pad>', '<unk>']
vocab_tokens = specials + [tok for tok, _ in token_freq.most_common(MAX_VOCAB - len(specials))]

# This is the key line:
word2idx = {word: idx for idx, word in enumerate(vocab_tokens)}

# Special token indices (needed for padding & unknowns)
pad_idx = word2idx['<pad>']
unk_idx = word2idx['<unk>']

### Encode and pad tweets

***every sample has exactly 50 tokens(MAX_LEN)***
and Zeros (0) represent <pad>

In [42]:
tweet_lengths = X_train['Final_Cleaned_Tweet'].astype(str).apply(lambda x: len(x.split()))
print("Max tweet length:", tweet_lengths.max())
print("95th percentile:", tweet_lengths.quantile(0.95))
print("99th percentile:", tweet_lengths.quantile(0.99))
print("Average:", tweet_lengths.mean())

Max tweet length: 108
95th percentile: 26.0
99th percentile: 30.0
Average: 14.21446636317653


***99% of tweets are ≤ 30 tokens long, and almost zero truncation (only 1% of tweets are longer)***

In [ ]:
MAX_LEN = 32 #  Maximum length of input sequences   

def encode(text):
    tokens = text.split()  # split string to tokens here
    ids = [word2idx.get(tok, unk_idx) for tok in tokens]
    if len(ids) < MAX_LEN:
        ids += [pad_idx] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]
    return ids


In [44]:
X_train_encoded = X_train['Final_Cleaned_Tweet'].astype(str).apply(encode).tolist()

In [45]:
print("Encoded input shape (train):", len(X_train_encoded), "x", len(X_train_encoded[0]))
print("First encoded tweet (IDs):", X_train_encoded[0])

Encoded input shape (train): 183861 x 30
First encoded tweet (IDs): [6, 16, 295, 18324, 1180, 146, 778, 3, 2, 8297, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [46]:
X_val_encoded  = X_val['Final_Cleaned_Tweet'].astype(str).apply(encode).tolist()
X_test_encoded = X_test['Final_Cleaned_Tweet'].astype(str).apply(encode).tolist()

In [47]:
y_train.sample(5)

71502      neutral
373       negative
158212    positive
30350     negative
39125     negative
Name: Sentiment, dtype: object

### Define label encoding

In [48]:
label2id = {'negative': 0, 'neutral': 1, 'positive': 2}

### Apply mapping to all label splits

In [49]:
y_train = y_train.astype(str).map(label2id)
y_val   = y_val.astype(str).map(label2id)
y_test  = y_test.astype(str).map(label2id)

In [50]:
print("Train label counts:\n", y_train.value_counts())
print("Val label counts:\n", y_val.value_counts())
print("Test label counts:\n", y_test.value_counts())

Train label counts:
 Sentiment
0    61287
1    61287
2    61287
Name: count, dtype: int64
Val label counts:
 Sentiment
0    5000
1    5000
2    5000
Name: count, dtype: int64
Test label counts:
 Sentiment
2    13221
1     3416
0     3363
Name: count, dtype: int64


### convert to tensors

In [51]:
X_train_tensor = torch.tensor(X_train_encoded)
X_val_tensor   = torch.tensor(X_val_encoded)
X_test_tensor  = torch.tensor(X_test_encoded)

y_train_tensor = torch.tensor(y_train.values).long()
y_val_tensor   = torch.tensor(y_val.values).long()
y_test_tensor  = torch.tensor(y_test.values).long()

In [52]:
print("X_train_tensor shape:", X_train_tensor.shape)  # Expected: (num_samples, MAX_LEN)
print("y_train_tensor shape:", y_train_tensor.shape)  # Expected: (num_samples, MAX_LEN) # Expected: (num_samples,)

X_train_tensor shape: torch.Size([183861, 30])
y_train_tensor shape: torch.Size([183861])


In [53]:
print("X_val_tensor shape:", X_val_tensor.shape)
print("y_val_tensor shape:", y_val_tensor.shape)

print("X_test_tensor shape:", X_test_tensor.shape)
print("y_test_tensor shape:", y_test_tensor.shape)

X_val_tensor shape: torch.Size([15000, 30])
y_val_tensor shape: torch.Size([15000])
X_test_tensor shape: torch.Size([20000, 30])
y_test_tensor shape: torch.Size([20000])


In [54]:
print("Unique labels in train:", torch.unique(y_train_tensor))

Unique labels in train: tensor([0, 1, 2])


### Dataset and DataLoader setup
***batched and padded token ID tensors with labels***

In [55]:
from torch.utils.data import Dataset, DataLoader

class TweetDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TweetDataset(X_train_tensor, y_train_tensor)
val_dataset   = TweetDataset(X_val_tensor, y_val_tensor)
test_dataset  = TweetDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=128)
test_loader  = DataLoader(test_dataset, batch_size=128)


In [56]:
batch = next(iter(train_loader))
print("Batch X shape:", batch[0].shape)  # Expected: [batch_size, MAX_LEN]
print("Batch y shape:", batch[1].shape)  # Expected: [batch_size]


Batch X shape: torch.Size([128, 30])
Batch y shape: torch.Size([128])


### Saving files contains padded, tokenized tweet tensors and encoded labels.

In [59]:
np.save('X_train_encoded.npy', X_train_tensor.cpu().numpy())
np.save('X_val_encoded.npy', X_val_tensor.cpu().numpy())
np.save('X_test_encoded.npy', X_test_tensor.cpu().numpy())

np.save('y_train.npy', y_train_tensor.cpu().numpy())
np.save('y_val.npy', y_val_tensor.cpu().numpy())
np.save('y_test.npy', y_test_tensor.cpu().numpy())


### Vocabulary Mapping
- For embedding layer size and any reverse lookup.

In [60]:
import json

with open('word2idx.json', 'w') as f:
    json.dump(word2idx, f)


### Special Token Indexes and Config

In [61]:
config = {
    "pad_idx": pad_idx,
    "unk_idx": unk_idx,
    "max_len": MAX_LEN,
    "vocab_size": len(word2idx)
}

with open('config.json', 'w') as f:
    json.dump(config, f)
